# Prompt 16: Practice Exam Questions — All Objectives

**HashiCorp Certified Terraform Associate (004) — Full Practice Exam**

25 realistic questions covering all 8 exam objectives. Click the **Answer** disclosure to reveal each answer and explanation.

---

## Distribution

| Objective | Topic | Questions |
|---|---|---|
| 1 | IaC Concepts | Q1–Q3 |
| 2 | Terraform Fundamentals (providers, state, plugin model) | Q4–Q6 |
| 3 | Core Terraform Workflow (CLI commands) | Q7–Q10 |
| 4 | Terraform Configuration (HCL — resources, variables, functions, lifecycle) | Q11–Q16 |
| 5 | Terraform Modules | Q17–Q18 |
| 6 | State Management and Backends | Q19–Q21 |
| 7 | Import and Logging | Q22–Q23 |
| 8 | HCP Terraform | Q24–Q25 |

---

## Objective 1 — IaC Concepts (Q1–Q3)

**Q1: Which of the following best describes the primary benefit of Infrastructure as Code (IaC) compared to manually provisioning infrastructure through a cloud console?**

A) IaC tools are faster at provisioning resources because they use internal cloud provider APIs unavailable through the console  
B) IaC allows infrastructure to be version-controlled, reviewed, tested, and reproduced consistently  
C) IaC eliminates the need for cloud provider accounts and reduces cost by running locally  
D) IaC guarantees that infrastructure will be identical across all cloud providers without any configuration changes  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The core benefit of IaC is that infrastructure definitions live in code files — they can be committed to version control, reviewed via pull requests, tested in CI/CD pipelines, and reliably reproduced across environments. **A** is false — IaC tools use the same APIs as the console. **C** is false — you still need cloud accounts and IaC doesn't eliminate costs. **D** is false — cloud providers have different resource types and you do need provider-specific configuration.

</details>

---

**Q2: Terraform uses a declarative approach to infrastructure management. What does this mean?**

A) You write scripts that describe each individual API call needed to create infrastructure step by step  
B) You describe the desired end state of your infrastructure and Terraform determines how to achieve it  
C) Terraform runs infrastructure scripts imperatively and reports which commands succeeded  
D) You declare which team members are allowed to modify infrastructure and Terraform enforces access control  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: In a **declarative** model, you specify *what* you want (an EC2 instance with these properties), not *how* to create it (step-by-step API calls). Terraform calculates the difference between current state and desired state and takes the necessary actions. This contrasts with **imperative** (procedural) tools like Ansible scripts that specify each step. **A** and **C** describe imperative approaches.

</details>

---

**Q3: True or False — Terraform is a cloud-agnostic tool, meaning the same Terraform configuration files (.tf files) work identically across AWS, Azure, and GCP with no modifications required.**

A) True — Terraform abstracts all cloud providers, so configurations are fully portable  
B) False — Terraform supports multiple cloud providers through separate provider plugins, but resource types, arguments, and configurations differ between providers  

<details><summary>Answer</summary>

**Answer: B — False**

**Explanation**: Terraform is cloud-agnostic in the sense that it supports many providers through the plugin model, but each provider has its own resource types and arguments. An `aws_instance` resource is completely different from `azurerm_virtual_machine` or `google_compute_instance`. Terraform's HCL language and workflow are consistent, but the actual resource configurations are provider-specific. You cannot run the same `.tf` files unchanged across different cloud providers.

</details>

---

## Objective 2 — Terraform Fundamentals (Q4–Q6)

**Q4: Where does Terraform download provider plugins during `terraform init`?**

A) Into the system's global PATH directory so all Terraform configurations can share them  
B) Into the `.terraform/providers/` directory within the working directory of the configuration  
C) Into `~/.terraform.d/plugins/` regardless of the working directory  
D) Into a shared cache that all Terraform configurations on the machine automatically use  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: `terraform init` downloads providers into `.terraform/providers/` inside the current working directory. This directory is local to that specific configuration. **C** is partially related — `~/.terraform.d/plugins/` is the legacy manual plugin directory and can serve as a local mirror/cache, but it is not the default download destination. **A** is false — providers are not placed in PATH. **D** is achievable with an explicit provider cache directory (`TF_PLUGIN_CACHE_DIR`) but is not the default behavior.

</details>

---

**Q5: What is the purpose of the `.terraform.lock.hcl` file?**

A) It prevents multiple team members from running `terraform apply` at the same time  
B) It records the exact provider versions selected during `terraform init` so that subsequent inits use the same versions  
C) It stores the Terraform state file checksum to detect unauthorized changes to state  
D) It locks the Terraform binary version to ensure all team members use the same version of Terraform CLI  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The **dependency lock file** (`.terraform.lock.hcl`) records the exact provider versions (and their checksums) that were selected when `terraform init` was run. On subsequent inits, Terraform uses the locked versions rather than re-resolving constraints. This ensures consistent provider versions across team members and CI runs. It should be **committed to version control**. **A** describes state locking. **C** is not a real feature. **D** describes the `required_version` constraint in the `terraform {}` block.

</details>

---

**Q6: A developer needs to use two different AWS accounts in the same Terraform configuration — one for production resources and one for logging resources. How should this be configured?**

A) Create two separate Terraform configurations and use `terraform_remote_state` to share outputs between them  
B) Use `provider "aws" { alias = "logging" ... }` to create a second AWS provider configuration, then reference it with `provider = aws.logging` on logging resources  
C) Use `workspace` to switch between provider configurations during apply  
D) This is not supported — Terraform configurations can only connect to one AWS account  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The `alias` meta-argument on a `provider` block creates a second named configuration of the same provider type. Resources that need to use the non-default provider configuration specify `provider = aws.logging` (using the alias). This is the canonical way to deploy resources across multiple accounts, regions, or with different credentials in a single configuration. **A** works but is overly complex when the alias approach exists. **C** is incorrect — workspaces manage state isolation, not provider configuration switching.

</details>

---

## Objective 3 — Core Terraform Workflow (Q7–Q10)

**Q7: A developer runs `terraform validate` and it succeeds. What does this guarantee?**

A) The configuration is syntactically valid and all referenced resources exist in the cloud provider account  
B) The configuration is syntactically valid and internally consistent — required arguments are present, types are correct, and references are valid  
C) The configuration will apply successfully without errors  
D) The configuration has been checked against the provider's live API and all resource arguments are valid  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: `terraform validate` checks the **local configuration only** — syntax, required arguments, type constraints, and whether references resolve. It does **not** contact cloud provider APIs. A configuration can pass `validate` and still fail during `apply` if cloud-side constraints are violated (e.g., an AMI ID doesn't exist, or an IAM role lacks required permissions). **A**, **C**, and **D** all incorrectly imply that `validate` contacts cloud APIs or guarantees successful apply.

</details>

---

**Q8: What is the correct sequence of commands for the standard Terraform workflow when creating infrastructure for the first time?**

A) `terraform plan` → `terraform init` → `terraform apply`  
B) `terraform init` → `terraform validate` → `terraform plan` → `terraform apply`  
C) `terraform init` → `terraform apply` → `terraform plan`  
D) `terraform validate` → `terraform init` → `terraform apply`  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The standard workflow is: **init** (download providers/modules, set up backend) → **validate** (check configuration correctness) → **plan** (preview changes) → **apply** (execute changes). `terraform init` must always run first — without it, providers aren't downloaded and `plan` cannot execute. While `validate` is technically optional (plan will catch errors too), it's part of the best-practice workflow. **A** has init after plan which is impossible. **C** skips plan. **D** validates before init which would fail.

</details>

---

**Q9: What does `terraform plan -refresh-only` do?**

A) Runs a plan that refreshes provider plugin versions and downloads updates  
B) Creates a plan that only updates Terraform state to match the real infrastructure — it does not plan any configuration changes  
C) Applies all pending changes to infrastructure and then refreshes the state file  
D) Queries the cloud provider and outputs the current state of all resources without modifying state  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: `terraform plan -refresh-only` generates a plan whose only effect is updating the Terraform **state file** to reflect actual infrastructure. This is used for **drift detection** — if someone changed a resource outside Terraform, this command shows what those changes are. It does NOT plan to apply your configuration changes. To execute the state update, run `terraform apply -refresh-only`. **A** is wrong — it has nothing to do with provider versions. **C** is wrong — plan never modifies anything; it only previews. **D** describes the refresh behavior but the key distinction is that `-refresh-only` produces a *plan* (which can then be applied to update state).

</details>

---

**Q10: True or False — `terraform destroy` is equivalent to running `terraform apply` with every resource marked for deletion.**

A) True — `terraform destroy` generates a destroy plan for all resources and applies it  
B) False — `terraform destroy` uses a separate internal code path that is completely different from `terraform apply`  

<details><summary>Answer</summary>

**Answer: A — True**

**Explanation**: `terraform destroy` is indeed equivalent to `terraform apply -destroy`. It generates a plan where every resource in the state is marked for deletion, then applies that plan. In fact, `terraform apply -destroy` and `terraform destroy` are functionally identical. The destroy plan can be saved with `terraform plan -destroy -out=destroy.tfplan` and then applied with `terraform apply destroy.tfplan`. This reinforces that `destroy` is just a special case of `apply`.

</details>

---

## Objective 4 — Terraform Configuration / HCL (Q11–Q16)

**Q11: Given the following configuration, what will `terraform output instance_ids` return after a successful apply?**

```hcl
variable "names" {
  default = ["web", "api", "worker"]
}

resource "random_id" "ids" {
  for_each    = toset(var.names)
  byte_length = 4
}

output "instance_ids" {
  value = { for k, v in random_id.ids : k => v.hex }
}
```

A) A list of three hex strings: `["aabb", "ccdd", "eeff"]`  
B) A map where keys are the names (`"web"`, `"api"`, `"worker"`) and values are the hex IDs  
C) An error — `for_each` with a list is not supported; `toset()` is invalid here  
D) A single hex string because `for_each` collapses multiple resources into one  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: `toset(var.names)` converts the list `["web", "api", "worker"]` to a set, which is valid for `for_each`. This creates three `random_id` resource instances keyed by the set values. The `for` expression `{ for k, v in random_id.ids : k => v.hex }` produces a **map** where each key is the instance key (`"web"`, `"api"`, `"worker"`) and each value is the `.hex` attribute of that instance. **A** is wrong — the result is a map, not a list. **C** is wrong — `toset()` is the correct way to use a list with `for_each`. **D** is wrong — `for_each` creates multiple instances, not a single one.

</details>

---

**Q12: What is the difference between a Terraform `variable` and a Terraform `local`?**

A) Variables store sensitive data; locals store non-sensitive computed values  
B) Variables are inputs to a module that can be set by the caller; locals are values computed within the module and cannot be overridden from outside  
C) Locals are evaluated at plan time while variables are evaluated at apply time  
D) Variables require a type constraint; locals do not support type constraints  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: `variable` blocks define inputs to a module — their values can be set by the caller (via `-var`, `.tfvars` files, environment variables, or the module block for child modules). `local` values are computed within the module from other expressions and cannot be set externally. They are for avoiding repetition and expressing intermediate computations. **A** is wrong — both variables and locals can hold any data type; sensitive is a separate attribute. **C** is wrong — both are evaluated at plan time. **D** is wrong — both can have or omit type constraints.

</details>

---

**Q13: What will the following expression evaluate to?**

```hcl
locals {
  result = join("-", ["us", "east", "1"])
}
```

A) `["us", "east", "1"]`  
B) `"us-east-1"`  
C) `"us, east, 1"`  
D) An error — `join()` requires a set, not a list  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The `join(separator, list)` built-in function concatenates list elements into a single string using the separator. `join("-", ["us", "east", "1"])` produces `"us-east-1"`. **A** is the input, not the output. **C** would be produced by `join(", ", ...)`. **D** is wrong — `join()` works with lists (and sets); there is no requirement for a set.

</details>

---

**Q14: A resource has the following lifecycle block. What will happen if you run `terraform destroy`?**

```hcl
resource "aws_db_instance" "main" {
  # ... other arguments
  lifecycle {
    prevent_destroy = true
  }
}
```

A) The database will be destroyed successfully — `prevent_destroy` only applies to `terraform apply`, not `terraform destroy`  
B) Terraform will display an error and refuse to execute the destroy plan because `prevent_destroy = true` is set  
C) Terraform will warn about `prevent_destroy` but continue with the destruction after user confirmation  
D) `prevent_destroy = true` will cause Terraform to take a snapshot of the database before deleting it  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: `prevent_destroy = true` causes Terraform to **error and abort** any plan that would destroy that resource — including `terraform destroy`. The error appears during the plan phase, before any changes are applied. The only way to destroy the resource is to first remove or set `prevent_destroy = false` in the configuration, then run `terraform apply`. **A** is wrong — it applies to any plan that includes a destroy action. **C** is wrong — there is no override confirmation; it's a hard stop. **D** is wrong — `prevent_destroy` has no snapshot functionality.

</details>

---

**Q15: What does the `depends_on` meta-argument do, and when should it be used?**

A) It defines the order in which Terraform configuration files are loaded — use it to control file parsing order  
B) It creates an explicit dependency between resources when Terraform cannot detect the dependency automatically from configuration references  
C) It replaces implicit dependencies — when `depends_on` is specified, Terraform ignores all other dependencies for that resource  
D) It is required for every resource that uses outputs from another resource  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: Terraform automatically detects dependencies from attribute references (e.g., `resource.type.name.attribute`). `depends_on` is for **hidden dependencies** that cannot be inferred from configuration — for example, an application server that must wait for an IAM policy to be attached even though the server resource doesn't reference the policy's attributes directly. Use it sparingly; overuse can prevent parallel execution and make configurations harder to understand. **A** is wrong — it has nothing to do with file loading order. **C** is wrong — `depends_on` adds to implicit dependencies, it doesn't replace them. **D** is wrong — if a reference exists in configuration, `depends_on` is redundant.

</details>

---

**Q16: A developer wants to ensure that a new security group is created before the old one is destroyed during a Terraform update. Which lifecycle argument accomplishes this?**

A) `ignore_changes = ["all"]`  
B) `prevent_destroy = true`  
C) `create_before_destroy = true`  
D) `replace_triggered_by = [aws_security_group.old]`  

<details><summary>Answer</summary>

**Answer: C**

**Explanation**: `create_before_destroy = true` changes Terraform's default destroy-then-create behavior. Normally, when a resource must be replaced, Terraform destroys the old resource first, then creates the new one. With `create_before_destroy = true`, Terraform provisions the new resource first, then destroys the old one — crucial when you can't have a gap in availability. **A** ignores attribute changes and prevents updates. **B** blocks all destruction. **D** triggers replacement when a referenced resource changes — it doesn't control the order of create/destroy during that replacement (though combining it with `create_before_destroy = true` is a valid pattern).

</details>

---

## Objective 5 — Terraform Modules (Q17–Q18)

**Q17: A team wants to use a module from the public Terraform Registry. Which of the following `source` values is correctly formatted?**

A) `source = "https://registry.terraform.io/modules/hashicorp/consul/aws"`  
B) `source = "hashicorp/consul/aws"`  
C) `source = "registry.terraform.io:hashicorp/consul/aws"`  
D) `source = "terraform-aws-modules/vpc"`  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The public Terraform Registry source format is `<NAMESPACE>/<MODULE>/<PROVIDER>` — no URL prefix, no protocol, no port. Terraform implicitly resolves this to `registry.terraform.io`. The correct source for the HashiCorp Consul module for AWS is `"hashicorp/consul/aws"`. **A** uses a full URL which is not valid for registry sources (that format is for HTTP archive sources). **C** uses a colon separator which is invalid syntax. **D** is missing the provider component — the format requires all three parts.

</details>

---

**Q18: A root module calls a child module and the child module declares an output named `instance_ip`. How does the root module access that output?**

A) `var.instance_ip`  
B) `output.web_server.instance_ip` (where `web_server` is the module block name)  
C) `module.web_server.instance_ip`  
D) The output is automatically available as a variable in the root module  

<details><summary>Answer</summary>

**Answer: C**

**Explanation**: Child module outputs are accessed via `module.<module_block_name>.<output_name>`. If the module block is named `web_server` and the child module declares `output "instance_ip"`, the root module accesses it as `module.web_server.instance_ip`. **A** is wrong — `var.` accesses input variables. **B** uses `output.` prefix which is not the correct syntax for module outputs. **D** is wrong — there is no automatic inheritance; outputs must be explicitly accessed through the `module.<name>.<output>` syntax.

</details>

---

## Objective 6 — State Management and Backends (Q19–Q21)

**Q19: A team is configuring the S3 backend for state storage. Which additional AWS service must they provision to enable state locking?**

A) Amazon SQS — to queue state updates and prevent concurrent writes  
B) Amazon ElastiCache — to cache state in memory for fast access  
C) Amazon DynamoDB — a table is used as a distributed lock mechanism for the S3 backend  
D) AWS Lambda — to enforce locking rules when state is written  

<details><summary>Answer</summary>

**Answer: C**

**Explanation**: The S3 backend stores state in an S3 bucket but S3 itself has no native locking mechanism suitable for Terraform. To enable **state locking**, a **DynamoDB table** is used — Terraform writes a lock record to DynamoDB before writing to S3, and removes it after. The `dynamodb_table` argument in the backend configuration enables this. All other options (SQS, ElastiCache, Lambda) have no role in S3 backend state locking.

</details>

---

**Q20: A Terraform apply operation is interrupted mid-run due to a network failure. The state is now locked. What should an operator do?**

A) Delete the state file and start fresh — there's no safe way to recover from a stuck lock  
B) Run `terraform force-unlock <lock-id>` only after confirming that no other Terraform operation is actually running  
C) Run `terraform force-unlock <lock-id>` immediately — it's always safe to force-unlock  
D) Contact AWS support to release the DynamoDB lock record  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: `terraform force-unlock <lock-id>` releases a stuck lock — but it must be used with caution. If another Terraform operation is actively running and holding the lock, force-unlocking while it runs can cause state corruption. The correct procedure is: first **verify** no operation is actually running (check CI/CD pipelines, other terminals), then use `force-unlock`. The Lock ID is provided in the error message when a lock is present. **A** is destructive and unnecessary. **C** is dangerous — forcing an unlock while an operation runs corrupts state. **D** is wrong — you can release the DynamoDB lock yourself via `force-unlock`.

</details>

---

**Q21: True or False — The `backend` block inside `terraform {}` can use variable references to make the backend configuration dynamic. For example: `bucket = var.state_bucket`**

A) True — variables can be used anywhere in Terraform configuration, including backend blocks  
B) False — backend configuration is processed before variables are available, so variable references are not allowed in backend blocks  

<details><summary>Answer</summary>

**Answer: B — False**

**Explanation**: Backend configuration is evaluated during `terraform init`, which runs **before** variables are processed. Therefore, backend blocks cannot contain variable references, local references, or most expressions — only literal values are supported. If you need to vary backend configuration (e.g., different S3 buckets per environment), use **partial backend configuration** with `-backend-config` CLI flags or separate `.tfbackend` files passed at init time. This is a common exam trap — the restriction applies specifically to `backend {}` blocks.

</details>

---

## Objective 7 — Import and Logging (Q22–Q23)

**Q22: A developer runs `terraform import aws_instance.web i-1234567890abcdef0`. What is the result?**

A) Terraform fetches the EC2 instance from AWS, generates the HCL configuration block for it, and adds it to `main.tf`  
B) Terraform writes the EC2 instance's attributes to the state file but does NOT generate any HCL configuration  
C) Terraform plans the import, shows a preview, and waits for confirmation before writing to state  
D) Terraform destroys the existing EC2 instance and recreates it under Terraform management  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The **legacy `terraform import` CLI command** writes the resource to **state only** — it does not generate HCL configuration. The developer must have already written an `aws_instance "web"` resource block before running the import command, or `terraform plan` will show that the resource needs to be destroyed (since it's in state but not in config). Configuration generation is a feature of the newer **`import` block** approach (Terraform 1.5+) combined with `terraform plan -generate-config-out=generated.tf`. **A** describes `import` block behavior. **C** is wrong — the CLI import command writes directly to state with no plan/confirm cycle.

</details>

---

**Q23: A developer is troubleshooting why a provider is making unexpected API calls. Which environment variable and value provides the most detailed logging?**

A) `TF_LOG=WARN`  
B) `TF_LOG=ERROR`  
C) `TF_LOG=DEBUG`  
D) `TF_LOG=TRACE`  

<details><summary>Answer</summary>

**Answer: D**

**Explanation**: `TF_LOG` levels in order of increasing verbosity: `ERROR` → `WARN` → `INFO` → `DEBUG` → `TRACE`. **TRACE** is the most verbose level and shows every provider API call, request/response bodies, and internal Terraform core decisions. It's the best option for understanding exactly what API calls a provider is making. **DEBUG** is one step below TRACE and is useful for provider-level troubleshooting but may miss some low-level details. For the exam, **TRACE** = maximum verbosity = most detail.

</details>

---

## Objective 8 — HCP Terraform (Q24–Q25)

**Q24: Which statement correctly describes the difference between the `cloud` block and `backend "remote"` for connecting to HCP Terraform?**

A) `backend "remote"` is the current recommended approach; the `cloud` block is deprecated  
B) The `cloud` block (available since Terraform 1.1) is the preferred approach; it supports workspace tag-based filtering and is more feature-complete. `backend "remote"` still works but is deprecated for HCP Terraform usage  
C) They are functionally identical — use either based on team preference  
D) The `cloud` block only works for local execution mode; `backend "remote"` supports all execution modes  

<details><summary>Answer</summary>

**Answer: B**

**Explanation**: The `cloud` block was introduced in Terraform 1.1 as the purpose-built way to configure HCP Terraform. It supports features that `backend "remote"` does not, including workspace **tag-based filtering** (connecting a CLI config to multiple workspaces matching a tag). HashiCorp recommends using the `cloud` block for all new HCP Terraform configurations and has deprecated `backend "remote"` for this purpose. **A** has it backwards. **C** is wrong — they are not identical; the `cloud` block has additional capabilities. **D** is wrong — the `cloud` block supports all execution modes including remote, local, and agent.

</details>

---

**Q25: An organization has a Sentinel policy with `enforcement_level = "hard-mandatory"` that blocks creation of EC2 instances with `instance_type` set to anything larger than `t3.large`. A critical production incident requires an immediate deployment with a `c5.2xlarge` instance. What can the organization do?**

A) An organization owner can temporarily override the hard-mandatory policy for this specific run  
B) The policy must be changed to `soft-mandatory` or `advisory`, the run executed, then changed back to `hard-mandatory`  
C) Hard-mandatory policies cannot be overridden by anyone — the instance type in the configuration must be changed to comply with the policy  
D) A team with Admin workspace permissions can override hard-mandatory policies on workspaces they administer  

<details><summary>Answer</summary>

**Answer: B or C**

**Explanation**: The exam answer is **C** — hard-mandatory policies **cannot be overridden by anyone**, including organization owners. However, since the policy is organization-controlled (not a read-only external policy set), an owner *could* edit the policy itself to change the enforcement level, run the apply, and change it back — this is what **B** describes. For exam purposes, **C** is the tested answer: hard-mandatory = no override by anyone. The distinction the exam tests is: **soft-mandatory** can be overridden by owners; **hard-mandatory** cannot be overridden — the only recourse is to modify the configuration to comply, or (as an owner) modify the policy definition itself. **A** is wrong — hard-mandatory explicitly cannot be overridden even by owners. **D** is wrong — workspace admin permissions don't override policy enforcement levels.

</details>

---

## Real-World Scenarios

<details><summary>Scenario 1: The Stuck State Lock in Production</summary>

**Situation**: A senior DevOps engineer's laptop loses network connectivity mid-apply. The pipeline shows:

```
Error acquiring the state lock: ConditionalCheckFailedException
Lock Info:
  ID:        e5f7a8b2-1234-5678-abcd-ef0123456789
  Path:      my-company-tf-state/prod/terraform.tfstate
  Operation: OperationTypeApply
  Who:       jane.doe@company.com
  Created:   2026-04-23 14:32:00 UTC
```

A newer CI/CD pipeline run is blocked waiting for the lock.

**The Exam Question it Raises**: What is the CORRECT procedure for dealing with a stuck Terraform state lock?

**Correct Answer**: Run `terraform force-unlock e5f7a8b2-1234-5678-abcd-ef0123456789` **ONLY after verifying** that Jane's apply is not still running (check if her laptop is back online, check if any apply process is running in the background).

```bash
# Step 1: Verify the apply process is truly dead
# Check CI/CD logs, ask Jane if her process is running

# Step 2: Force-unlock using the Lock ID from the error
terraform force-unlock e5f7a8b2-1234-5678-abcd-ef0123456789

# Terraform confirms:
# Do you really want to force-unlock?
#   Terraform will remove the lock on the remote state.
# yes
# Terraform state has been successfully unlocked!

# Step 3: Run terraform plan to verify state consistency
terraform plan
```

**Why the Other Options Are Wrong**:
- Deleting the DynamoDB lock record manually bypasses Terraform's safety checks
- Deleting and re-creating the state file loses all state history
- Running `force-unlock` immediately without checking if the process is still running risks state corruption

**Exam Sub-Objective**: State locking — `terraform force-unlock`, when to use, and cautions (Objective 6).

</details>

---

<details><summary>Scenario 2: The Validate Passes but Apply Fails</summary>

**Situation**: A junior engineer runs `terraform validate` on a new EC2 configuration and sees:

```
Success! The configuration is valid.
```

Confident, they run `terraform apply` and get:

```
Error: InvalidAMIID.NotFound: The image id '[ami-99999999]' does not exist
```

The engineer is confused — if validate passed, why did apply fail?

**The Exam Question it Raises**: What does `terraform validate` actually check?

**Correct Answer**: `terraform validate` only checks **local configuration correctness** — syntax, required arguments, type constraints, and internal references. It does **not** contact cloud provider APIs. The AMI ID `ami-99999999` is syntactically valid (it's a properly formatted string), so validate passes. Only when `apply` contacts the AWS EC2 API does AWS return the error that the AMI doesn't exist.

**Lesson**:
```
terraform validate  →  Local check only. Fast, no API calls.
terraform plan      →  Contacts provider API for data sources, checks IAM permissions
terraform apply     →  Creates/modifies/destroys actual resources
```

**Why This Matters**: The exam frequently tests whether candidates understand that `validate` does NOT contact cloud providers. A passing `validate` does not guarantee a successful `apply`.

**Exam Sub-Objective**: Understanding `terraform validate` scope and limitations (Objective 3).

</details>

---

<details><summary>Scenario 3: Module Output Access Confusion</summary>

**Situation**: A developer writes the following Terraform configuration:

```hcl
# root module — main.tf
module "database" {
  source = "./modules/rds"
  db_name = "production"
}

resource "aws_instance" "app" {
  # ... other config
  user_data = <<-EOT
    DB_HOST=${database.endpoint}  # Developer's attempt to reference module output
  EOT
}
```

The `rds` module declares `output "endpoint" { value = aws_db_instance.main.endpoint }`. The apply fails with a reference error.

**The Exam Question it Raises**: How do you correctly reference a module output from the root module?

**Correct Answer**: Use `module.<module_block_name>.<output_name>`:

```hcl
resource "aws_instance" "app" {
  user_data = <<-EOT
    DB_HOST=${module.database.endpoint}  # Correct syntax
  EOT
}
```

**Wrong Patterns and Why**:
- `database.endpoint` — missing `module.` prefix
- `var.database.endpoint` — `var.` is for input variables, not module outputs
- `output.database.endpoint` — `output.` prefix does not exist for module references
- `modules.database.endpoint` — `modules.` (plural) is not valid; it's `module.` (singular)

**Exam Sub-Objective**: Module variable scope — accessing child module outputs from parent module (Objective 5).

</details>

---

<details><summary>Scenario 4: Backend Configuration Variable Attempt</summary>

**Situation**: A platform engineer wants to make the S3 backend dynamic so different environments (dev, staging, prod) use different S3 buckets. They write:

```hcl
variable "environment" {
  default = "dev"
}

terraform {
  backend "s3" {
    bucket = "mycompany-tf-state-${var.environment}"  # Attempt to use variable
    key    = "terraform.tfstate"
    region = "us-east-1"
  }
}
```

Running `terraform init` fails with an error about the backend configuration.

**The Exam Question it Raises**: Why can't variables be used in backend configuration, and how should this be solved?

**Correct Answer**: Backend configuration is processed during `terraform init`, **before** variable values are available. Variable references in `backend {}` blocks are not supported.

**Solution — Partial Backend Configuration**:

```hcl
# main.tf — omit the dynamic values
terraform {
  backend "s3" {
    # bucket omitted — will be provided at init time
    key    = "terraform.tfstate"
    region = "us-east-1"
  }
}
```

```bash
# Provide the dynamic value at init time:
terraform init -backend-config="bucket=mycompany-tf-state-dev"

# Or use a per-environment .tfbackend file:
# dev.s3.tfbackend:
#   bucket = "mycompany-tf-state-dev"
terraform init -backend-config=dev.s3.tfbackend
```

**Exam Sub-Objective**: Backend configuration restrictions — no variable references; partial backend configuration with `-backend-config` (Objective 6).

</details>

---

<details><summary>Scenario 5: Choosing Between `import` Block and `terraform import` CLI</summary>

**Situation**: A new team is adopting Terraform to manage 50 existing EC2 instances that were manually created. A team member suggests using `terraform import aws_instance.web-01 i-1234567890abcdef0` for each instance. Another suggests using the `import` block. A discussion ensues about which approach is better and why.

**The Exam Question it Raises**: What is the key difference between `terraform import` CLI command and the `import` block (Terraform 1.5+)?

| Feature | `terraform import` CLI | `import` block |
|---|---|---|
| HCL generation | ❌ You write the resource block | ✓ Use `-generate-config-out=gen.tf` |
| Plan preview | ❌ Writes directly to state | ✓ Shows in `terraform plan` |
| Reviewable | ❌ No plan phase for the import | ✓ Team can review before apply |
| Preferred? | Legacy approach | ✓ Preferred since Terraform 1.5 |

**Using the `import` block (preferred)**:

```hcl
# For each of the 50 instances, add an import block:
import {
  to = aws_instance.web_01
  id = "i-1234567890abcdef0"
}
```

```bash
# Generate HCL configuration for the resource automatically:
terraform plan -generate-config-out=imported_instances.tf

# Review the generated configuration, then apply:
terraform apply
```

**Correct Answer for Exam**: The `import` block is the **preferred** approach because it:
1. Appears in `terraform plan` for review before committing
2. Can auto-generate HCL configuration with `-generate-config-out`
3. Is version-control friendly (import blocks live in `.tf` files)

The CLI `terraform import` command **still works** but does not generate configuration and bypasses the plan review step.

**Exam Sub-Objective**: `import` block vs CLI import — differences, when to use each, configuration generation (Objective 7).

</details>

---

## Quick-Reference Answer Key

| Question | Answer | Objective |
|---|---|---|
| Q1 | B | 1 — IaC Concepts |
| Q2 | B | 1 — IaC Concepts |
| Q3 | B (False) | 1 — IaC Concepts |
| Q4 | B | 2 — Providers |
| Q5 | B | 2 — Lock File |
| Q6 | B | 2 — Provider Alias |
| Q7 | B | 3 — validate |
| Q8 | B | 3 — Workflow |
| Q9 | B | 3 — refresh-only |
| Q10 | A (True) | 3 — destroy |
| Q11 | B | 4 — for_each + for expression |
| Q12 | B | 4 — variables vs locals |
| Q13 | B | 4 — join() function |
| Q14 | B | 4 — prevent_destroy |
| Q15 | B | 4 — depends_on |
| Q16 | C | 4 — create_before_destroy |
| Q17 | B | 5 — module source format |
| Q18 | C | 5 — module output access |
| Q19 | C | 6 — state locking / DynamoDB |
| Q20 | B | 6 — force-unlock |
| Q21 | B (False) | 6 — backend variables |
| Q22 | B | 7 — import CLI command |
| Q23 | D | 7 — TF_LOG levels |
| Q24 | B | 8 — cloud block vs backend remote |
| Q25 | C | 8 — hard-mandatory policy |